# 2. Structured Output and Usage

Validate model output with Pydantic and track tokens and latency.

In [ ]:
import json
import os
from typing import Literal

import boto3
from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [ ]:
api_key = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
if not api_key:
    raise RuntimeError("AWS_BEARER_TOKEN_BEDROCK environment variable is required")

In [ ]:
class DecisionOutput(BaseModel):
    model_config = ConfigDict(extra = 'forbid')

    decision: Literal[
        "recommend",
        "do_not_recommend",
        "need_more_information"
    ]

    summary: str
    reasons: list[str]
    questions: list[str]
    confidence: float = Field(ge=0.0 ,le=1.0)

In [ ]:
region = os.getenv("BEDROCK_REGION", "ap-south-1")
model_id = "google.gemma-3-4b-it"

client = boto3.client(
    service_name="bedrock-runtime",
    region_name=region,
)

In [ ]:
output_schema = json.dumps(
    DecisionOutput.model_json_schema(),
    indent=2
)

system_prompt = [
    {
        "text": f"""
        You are a personal decision assistant.

        Rules:
        - Ask questions when important information is missing.
        - Do not invent financial, weather, distance, or calendar data.
        - Return only a valid JSON object.
        - Do not use Markdown or code fences.
        - Keep Answer precise and with brief explanation.
        - If essential information is missing, decision must be "need_more_information".
        - Do not recommend a decision while asking essential questions.
        - You currently have no live weather, distance, calendar, or pricing tools.
        - Do not provide external facts unless they are supplied by the user or a tool.
        - Confidence must be low when required information is missing.
        - The JSON must follow this schema:

        {output_schema}
        """.strip()
    }
]


messages = []

session_usage = {
    "calls": 0,
    "input_tokens":0,
    "output_tokens":0,
    "total_tokens":0,
    "latency_ms":0
}

In [ ]:
def record_usage(response: dict) -> None: 
    usage = response["usage"]
    latency_ms = response.get("metrics",{}).get("latencyMs",0)

    session_usage["calls"] += 1
    session_usage["input_tokens"] += usage["inputTokens"]
    session_usage["output_tokens"] += usage["outputTokens"]
    session_usage["total_tokens"] += usage["totalTokens"]
    session_usage["latency_ms"] += latency_ms

    print("\nCall usage")
    print(f"Input tokens:  {usage['inputTokens']}")
    print(f"Output tokens: {usage['outputTokens']}")
    print(f"Total tokens:  {usage['totalTokens']}")
    print(f"Latency:       {latency_ms} ms")

    print("\nSession usage")
    print(f"Calls:         {session_usage['calls']}")
    print(f"Input tokens:  {session_usage['input_tokens']}")
    print(f"Output tokens: {session_usage['output_tokens']}")
    print(f"Total tokens:  {session_usage['total_tokens']}")
    print(f"Latency:       {session_usage['latency_ms']} ms")


def remove_code_fence(text: str) -> str:
    text = text.strip()

    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()
        return "\n".join(lines[1:-1]).strip()

    return text


In [ ]:
while True: 
    user_input = input("\nYou: ").strip()

    if user_input.lower() in {"exit", "quit"}:
        break

    # print(user_input)
    messages.append(
        {
            "role": "user",
            "content": [{"text": user_input}],
        }
    )

    response = client.converse(
        modelId = model_id,
        system = system_prompt,
        messages = messages,
        inferenceConfig= {
            "maxTokens": 500,
            "temperature": 0.2
        }
    )

    record_usage(response)
    stop_reason = response["stopReason"]
    print(f"Stop reason: {stop_reason}")

    if stop_reason == "max_tokens":
        messages.pop()
        print("The response was truncated. Increase maxTokens and try again.")
        continue

    assistant_message = response["output"]["message"]
    answer = "".join (
        block["text"]
        for block in assistant_message["content"]
        if "text" in block
    )

    print("Current input:", repr(user_input))

    for index, message in enumerate(messages):
        print(
            index,
            message["role"],
            [
                repr(block.get("text"))
                for block in message["content"]
            ],
        )

    clean_answer = remove_code_fence(answer)

    try:
        decision = DecisionOutput.model_validate_json(clean_answer)
    except ValidationError as error:
        print("\nInvalid model output")
        print(answer)
        print(error)
        continue

    messages.append(assistant_message)

    print("\nAssistant")
    print(decision.model_dump_json(indent = 2))
        